# 🥾 Side Trails · Step 2 — The Power Wall, SIMD and the Decision Tool
**A Billion Small Steps · Hardware for AI · Step 2 · Book sections 2.6, 2.7, App. B · Foothold lessons 2.6–2.7, 2.C**

Read the book sections first, then run this notebook from top to bottom. Runs on a **GPU**: Runtime → Change runtime type → **T4 GPU**.

| Part | What you do | Book |
|---|---|---|
| A | Clock speed and power | 2.6 (leaves 2.6 → rejoin 2.7) |
| B | Vector add: CPU vs GPU, from tiny to huge | 2.7 (leaves 2.7 → rejoin 2.8) |
| C | The CPU-or-GPU decision tool, versions 1 and 2 | App. B.1 (from 2.C) |

⏱️ About 1 hour. 📘 Textbook: H&P Ch 1 · Hamacher §12.2. Course site: https://aishugithub.github.io/course_GPU/

🥾 **Side Trails are optional.** Nothing in a later Core Lab needs them.

💬 **Every line of code has a comment** (after the `#`, or after `//` inside CUDA C). Read the comment, then the line it explains.

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run these cells first

In [ ]:
# Which versions is this session running? The book's printed outputs used Colab's free Tesla T4
# with PyTorch 2.14.1 (CUDA 13.0 build) on 8 Nov 2026; small differences in your numbers are normal.
# If a cell fails after a Colab update: remove the "#" at the start of the next line, run this cell,
# then Runtime → Restart session (slow; only when needed).
# !pip install torch==2.14.1
import platform                                    # platform: tells us the Python version
import numpy, torch                                # NumPy and PyTorch, to read their version numbers
print("Python ", platform.python_version())        # the Python version
print("NumPy  ", numpy.__version__)                # the NumPy version
print("PyTorch", torch.__version__)                # the PyTorch version (and its CUDA build, after the +)
try:                                               # CuPy is only used by the CUDA C kernel labs
    import cupy                                    # try to load it
    print("CuPy   ", cupy.__version__)             # its version
except Exception:                                  # not installed, or no GPU for it in this session
    print("CuPy    not available in this session") # that is fine for this notebook
if torch.cuda.is_available():                      # is there an NVIDIA GPU in this session?
    print("GPU    ", torch.cuda.get_device_name(0))  # yes: its name (the book used a Tesla T4)
else:                                              # no GPU
    print("GPU     none (CPU runtime)")            # say so

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

## Part A · Clock Speed and Power  *(Section 2.6)*

Is each GPU core faster than a CPU core? Ask the hardware: first the GPU, then the CPU.

In [ ]:
# nvidia-smi is the GPU's "task manager"; this asks it for the name, clock now and maximum (MHz),
# power now and its limit (watts), and temperature (°C). A "!" runs a system command, not Python.
!nvidia-smi --query-gpu=name,clocks.sm,clocks.max.sm,power.draw,power.limit,temperature.gpu --format=csv

In [ ]:
# the CPU side: its name and how many cores the operating system can use
import os, platform
print("CPU:", platform.processor() or platform.machine())   # the CPU's name, as the operating system reports it
print("Cores (logical):", os.cpu_count())   # how many cores the operating system can use

### ✏️ YOUR TURN (A)
| | CPU | GPU |
|---|---|---|
| Clock speed (MHz) | (Colab's CPU runs at about 2,000–2,200 MHz) | |
| Number of cores | | |
| Power limit (W) | — | |

The GPU's clock is **slower** than the CPU's. So where does its speed come from? (Section 2.6: many slow cores beat one fast core on throughput and on power.)

*Your answer:*

## Part B · Vector Add: CPU vs GPU  *(Section 2.7)*

`C = A + B` for arrays of growing size. On the GPU, every element is handled by its own thread (SIMT).
Here the data is created **directly on the GPU**, so we're timing only the arithmetic.

In [ ]:
# array sizes from 1 thousand to 5 crore numbers
sizes = [10**3, 10**4, 10**5, 10**6, 10**7, 5 * 10**7]
cpu_t, gpu_t = [], []   # the measured times go into these lists
for n in sizes:   # for each size
    # two arrays in CPU memory, and two made directly in GPU memory (so we time only the arithmetic)
    a_np, b_np = np.random.rand(n).astype(np.float32), np.random.rand(n).astype(np.float32)
    a_gpu, b_gpu = torch.rand(n, device=device), torch.rand(n, device=device)   # two arrays made directly in GPU memory
    # lambda makes a tiny unnamed function, so timeit can run the addition again and again
    cpu_t.append(timeit(lambda: a_np + b_np))
    gpu_t.append(timeit(lambda: a_gpu + b_gpu))   # the same addition on the GPU
    # [-1] is the time just measured; × 1e3 gives milliseconds
    print(f"n = {n:>11,d}   CPU {cpu_t[-1]*1e3:8.3f} ms   GPU {gpu_t[-1]*1e3:8.3f} ms   → GPU is {cpu_t[-1]/gpu_t[-1]:6.1f}× faster")

In [ ]:
plt.figure(figsize=(6, 4))                                   # a new 6 × 4 inch chart
plt.loglog(sizes, cpu_t, "o-", label="CPU (NumPy)")          # log scale on both axes; circles = CPU
plt.loglog(sizes, gpu_t, "s-", label=f"GPU ({device})")      # squares = GPU
plt.xlabel("array size n"); plt.ylabel("time (s)")           # axis labels
plt.legend(); plt.grid(alpha=0.3, which="both")              # colour key and faint grid
plt.title("Vector add: small = CPU wins, big = GPU wins")    # the chart's title
plt.show()                                                   # draw it

**Why `sync()`?** When Python asks the GPU to do something, it doesn't wait. It just queues the work and carries on (Section 4.4). Without `torch.cuda.synchronize()` you'd time how long it takes to *queue* the work, not to *do* it. Try deleting the `sync()` calls inside `timeit` (in the Setup cell) and re-running: the GPU will look impossibly fast.

### ✏️ YOUR TURN (B)
At roughly what array size does the GPU start to win? Why does the CPU win for tiny arrays?

*Your answer:*

---
## Part C · The decision tool in code  *(Appendix B.1, from 2.C)*
Version 1 always picks the GPU. Version 2 also knows the bytes and where the data is, using rough T4-class numbers.

In [ ]:
# version 1 of our decision tool
def decide(flops):
    # the GPU has more FLOPS, so always pick it
    return "GPU"
# what version 1 says about a tiny job
print("v1, 1,000 FLOPs:", decide(1_000))

In [ ]:
# version 2: it also knows the bytes, and where the data is
def decide(flops, nbytes, on_gpu):
    # the CPU: maths or memory, whichever is slower
    cpu = max(flops / 0.2e12, nbytes / 20e9)
    # the GPU: launch + copy (if the data isn't there yet) + the work itself
    gpu = 20e-6 + (0 if on_gpu else nbytes / 12e9) + max(flops / 4e12, nbytes / 300e9)
    # pick the faster machine
    return "CPU" if cpu < gpu else "GPU"
# a tiny job with its data still on the CPU
print("v2, tiny job, data on the CPU:", decide(1_000, 8_000, on_gpu=False))
# a 4,096 × 4,096 matrix multiply (2n³ FLOPs, three n × n FP32 matrices) with the data already on the GPU
print("v2, 4,096 matmul, data on the GPU:", decide(2 * 4096**3, 3 * 4096**2 * 4, on_gpu=True))

### ✏️ YOUR TURN (C)
Use version 2 on a ReLU over 10 crore numbers (1 FLOP and 8 bytes per number) with the data on the CPU. Which does it pick, and why?

*Your answer:*